# PEC Theta-Space Harmonic Analysis

Tests the theta-space PEC hypothesis from `docs/pec_theta_space_plan.md`: since the Polaris is an alt-az mount, periodic mechanical error should repeat with **motor/gear rotation angle (theta)**, not wall-clock time -- the RA/Dec<->motor mapping is orientation-dependent, and each motor's angular rate varies with where the mount is pointing, so a real worm-gear-driven periodic error smears out in the time domain but should stay sharp in the angle domain. This notebook is angle-domain first; the time-domain periodogram further down is kept only as a secondary cross-check, not the hypothesis itself.

Split out of `analyse_kf_pid.ipynb`/`analyse_pec.ipynb` into its own notebook, since this analysis has its own data-loading logic (auto-detects PECLOG vs SGLOG vs raw sync-guide residual lines, whichever the session actually has) independent of those notebooks' own KFLOG/PIDLOG- and PECLOG-plotting cells.

In [ ]:
import os, sys
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
import analyse_helpers
importlib.reload(kinematics)
importlib.reload(analyse_helpers)
from analyse_helpers import (
    resolve_log_files, load_pec, load_sglog, load_sync_guiding_residuals, load_kf_pid,
    find_site_location, find_tracking_segments,
    derive_theta_from_total_accum, derive_theta_from_sync_residuals, derive_theta_from_sglog,
    lombscargle_periodogram, periodogram_peak, periodogram_false_alarm_probability,
)

AXES = ["M1", "M2", "M3"]
MIN_SEGMENT_POINTS = 20   # below this a periodogram/FAP isn't meaningful (lombscargle_periodogram itself floors at 8)


# Load data

In [ ]:
# -- Choose correct log path (last set log_filenames is what is used) --------------------------------
# Single file:            'alpaca.log'                     (relative to LOG_DIR below)
# All rotated files:      'alpaca.log*'                    (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['alpaca.log.2', 'alpaca.log.1', 'alpaca.log']
# A full/relative path (as before) also still works and bypasses LOG_DIR: '../logs/alpaca.log'
LOG_DIR = '../logs/archive'   # base directory log_filenames below are resolved against

log_filenames = ['alpaca.soak_nopec_Beta4.3_08_31a*.log']  # PEC-off, sync-guide-on overnight capture -- no PECLOG, uses the sync-residual+KFLOG fallback path below
#LOG_DIR = '../logs/logs'
#log_filenames = ['alpaca.soak_pec_Beta4.3_08_29n*.log']    # PECLOG session (RLS PEC on) -- uses the PECLOG/total_accum path below
log_filenames = ['alpaca.soak_Beta4.4_09_01_sg_sglog_Dec10_a*.log']                 # sync guiding,     with logs(sg), Eagle Nebula


# Site location

In [ ]:
site = find_site_location(log_filenames, log_dir=LOG_DIR)
if site is None:
    # Not every driver version/config logs the 'Site lat = ... | lon = ...' line -- fall back
    # to this project's known site if it's missing from this particular capture.
    site = dict(lat_deg=-33.655075752841235, lon_deg=151.12181398189458)  # Sydney
    print("No 'Site lat = ...' line found in the log -- using fallback:", site)
else:
    print("Site location from log:", site)
LAT_DEG, LON_DEG = site['lat_deg'], site['lon_deg']


# Ground truth: `theta_true` per motor (PEC- and reset-independent)

Auto-detects which data source this session actually has and picks the matching reconstruction, in order of preference:

1. **PECLOG** (`load_pec` + `derive_theta_from_total_accum`) -- uses `total_accum`, never PEC-contaminated `resid` alone (see `docs/pec_theta_space_plan.md`'s Phase 0.1 correction: `resid` measures whatever error remains *after* PEC's own correction, so it systematically understates the true periodic error whenever PEC is active).
2. **SGLOG** (`load_sglog` + `derive_theta_from_sglog`) -- for a PEC-off session with `log_position` also off (no PECLOG, no KFLOG). Carries `theta_raw` and `total_accum` directly, no nearest-sample matching needed.
3. **Raw `SYNC GUIDING ... Residuals` lines + KFLOG** (`load_sync_guiding_residuals` + `load_kf_pid` + `derive_theta_from_sync_residuals`) -- fallback for a PEC-off session captured before SGLOG existed, where `log_position` had to be left on just to recover motor position.

Each source is split into independent segments (PECLOG: at every `n` PEC-model reset; SGLOG/sync-residual: at every `find_tracking_segments()` goto/slew/park boundary) before any periodogram is run on it -- concatenating across a reset/retarget would inject a large non-periodic discontinuity into a cumulative-motor-angle analysis.

In [ ]:
def _add_t_local(seg):
    seg = seg.reset_index(drop=True)
    seg['t_local_sec'] = (seg['timestamp'] - seg['timestamp'].iloc[0]).dt.total_seconds()
    return seg


def _split_by_n_reset(df):
    """Split a derive_theta_from_total_accum() result into per-PEC-segment sub-frames --
    each internal segment has its own RA/Dec anchor and LastPosition (see that function's
    docstring), so concatenating across a reset would inject a spurious discontinuity into
    any periodogram run across the seam."""
    n = df['n'].values
    starts = [0] + [i for i in range(1, len(df)) if n[i] < n[i - 1]]
    bounds = list(zip(starts, starts[1:] + [len(df)]))
    return [df.iloc[a:b] for a, b in bounds]


theta_cols = [f'theta_true_{i}' for i in (1, 2, 3)]


def _try_peclog():
    pec_df, _pec_config = load_pec(log_filenames, log_dir=LOG_DIR)
    pec_df = derive_theta_from_total_accum(pec_df, LAT_DEG, LON_DEG)
    segs = []
    for i, seg in enumerate(_split_by_n_reset(pec_df), start=1):
        seg = seg.dropna(subset=theta_cols)
        if len(seg) >= MIN_SEGMENT_POINTS:
            segs.append((f'segment {i}', _add_t_local(seg)))
    return 'PECLOG (total_accum)', segs


def _try_sglog():
    sg_df = load_sglog(log_filenames, log_dir=LOG_DIR)
    tsegs = find_tracking_segments(log_filenames, log_dir=LOG_DIR) or \
        [(sg_df.timestamp.min(), sg_df.timestamp.max(), None)]
    segs = []
    for i, (t0, t1, _dur) in enumerate(tsegs, start=1):
        sub = sg_df[(sg_df.timestamp >= t0) & (sg_df.timestamp <= t1)]
        if len(sub) < MIN_SEGMENT_POINTS:
            continue
        sub = derive_theta_from_sglog(sub, LAT_DEG, LON_DEG).dropna(subset=theta_cols)
        if len(sub) >= MIN_SEGMENT_POINTS:
            segs.append((f'segment {i}', _add_t_local(sub)))
    return 'SGLOG (total_accum)', segs


def _try_sync_residuals():
    resid_df = load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR)
    kf_df, _pid_df = load_kf_pid(log_filenames, log_dir=LOG_DIR)
    tsegs = find_tracking_segments(log_filenames, log_dir=LOG_DIR) or \
        [(resid_df.timestamp.min(), resid_df.timestamp.max(), None)]
    segs = []
    for i, (t0, t1, _dur) in enumerate(tsegs, start=1):
        sub = resid_df[(resid_df.timestamp >= t0) & (resid_df.timestamp <= t1)]
        if len(sub) < MIN_SEGMENT_POINTS:
            continue
        sub = derive_theta_from_sync_residuals(sub, kf_df, LAT_DEG, LON_DEG).dropna(subset=theta_cols)
        if len(sub) >= MIN_SEGMENT_POINTS:
            segs.append((f'segment {i}', _add_t_local(sub)))
    return 'SYNC GUIDING residual lines + KFLOG theta_meas_raw', segs


# Try each source in preference order. A source can be technically parseable but still
# useless (e.g. a PEC-off session that still has a couple of stray PECLOG rows from before
# PEC was disabled) -- fall through to the next source whenever one produces zero usable
# segments, not just on an outright load error.
source, segments = None, []
for _attempt, _name in [(_try_peclog, 'PECLOG'), (_try_sglog, 'SGLOG'),
                         (_try_sync_residuals, 'sync-residual+KFLOG')]:
    try:
        _src, _segs = _attempt()
    except (FileNotFoundError, ValueError) as e:
        print(f"{_name}: unavailable ({e})")
        continue
    if _segs:
        source, segments = _src, _segs
        break
    print(f"{_name}: parsed but produced no usable segments (>= {MIN_SEGMENT_POINTS} points) -- trying next source...")

if source is None:
    raise RuntimeError("No usable ground-truth source found (PECLOG, SGLOG, and sync-residual+"
                        "KFLOG all empty/unavailable) for this session.")

print(f"\nGround-truth source: {source}")
print(f"Usable segments (>={MIN_SEGMENT_POINTS} points each): {len(segments)}")
for label, seg in segments:
    dur_h = (seg['timestamp'].iloc[-1] - seg['timestamp'].iloc[0]).total_seconds() / 3600
    print(f"  {label}: n={len(seg)}, span={dur_h:.2f}h, {seg['timestamp'].iloc[0]} -> {seg['timestamp'].iloc[-1]}")


In [ ]:
seg

## Time Domain - meas-true theta vs timestamp

In [ ]:
fig = make_subplots(rows=len(AXES), cols=1, shared_xaxes=False,
    subplot_titles=[f'{ax}: angle-domain (all segments)' for ax in AXES],
    vertical_spacing=0.08)

for label, seg in segments:
    t = seg['t_local_sec'].values/60
    ts= seg['timestamp']
    for i, ax in enumerate(AXES, start=1):
        theta_true = seg[f'theta_true_{i}'].values
        theta_meas = seg[f'theta_meas_{i}'].values
        true_detrended = theta_true - np.polyval(np.polyfit(t, theta_true, 1), t)
        fig.add_trace(go.Scatter(x=ts, y=theta_meas-theta_true, mode='lines',
            name=f'{label} {ax} true'), row=i, col=1)
        
fig.update_xaxes(title_text='Period (degrees of motor rotation)', row=len(AXES), col=1)
for i in range(1, len(AXES) + 1):
    fig.update_yaxes(title_text='log10(power)', row=i, col=1)
fig.update_layout(height=900, width=1200, template='plotly_dark',
    title=f'Time-domain meas-true per motor -- source: {source}',
    hovermode='x unified')
fig.show()


## Angle Domain - meas-true theta vs angle

In [ ]:
fig = make_subplots(rows=len(AXES), cols=1, shared_xaxes=False,
    subplot_titles=[f'{ax}: angle-domain (all segments)' for ax in AXES],
    vertical_spacing=0.08)

for label, seg in segments:
    t = seg['t_local_sec'].values/60
    for i, ax in enumerate(AXES, start=1):
        theta_true = seg[f'theta_true_{i}'].values
        theta_meas = seg[f'theta_meas_{i}'].values
        true_detrended = theta_true - np.polyval(np.polyfit(t, theta_true, 1), t)
        fig.add_trace(go.Scatter(x=theta_meas, y=theta_meas-theta_true, mode='lines',
            name=f'{label} {ax} true'), row=i, col=1)
        
fig.update_xaxes(title_text='Period (degrees of motor rotation)', row=len(AXES), col=1)
for i in range(1, len(AXES) + 1):
    fig.update_yaxes(title_text='log10(power)', row=i, col=1)
fig.update_layout(height=900, width=1200, template='plotly_dark',
    title=f'Angle-domain meas-true per motor -- source: {source}',
    hovermode='x unified')
fig.show()


# Angle-domain periodogram per motor, per segment (the core test)

`x = theta_true_i` (the reconstructed, PEC/correction-independent drift trajectory in motor-angle space -- see above); `y` = that *same* `theta_true_i` series, but pre-detrended against **time** first, then paired with the raw (non-time-detrended) `x`.

This two-step detrend is required, not optional -- found and fixed during the original investigation (`docs/pec_theta_space_plan.md`): `theta_true_i` is dominated by the same large, intentional tracking motion in both `x` and `y` (it's literally the same array), so if `lombscargle_periodogram()`'s own linear detrend is left to run directly against `x` (`detrend=True`), it fits `y ≈ x` almost perfectly and zeroes out the entire signal -- a genuine degenerate case, not a hypothetical one. Detrending against time first isolates the periodic wobble; pairing that wobble with the raw (still-in-degrees) `x` is what lets the periodogram report the period in degrees of shaft rotation, the actual quantity this hypothesis is about.

In [ ]:
FAP_N_SHUFFLES = 200
rng = np.random.default_rng(42)

angle_results = []
for label, seg in segments:
    t = seg['t_local_sec'].values
    for i, ax in enumerate(AXES, start=1):
        x = seg[f'theta_true_{i}'].values
        y_detrended = x - np.polyval(np.polyfit(t, x, 1), t)
        try:
            fap, peak_period, peak_power = periodogram_false_alarm_probability(
                x, y_detrended, n_shuffles=FAP_N_SHUFFLES, rng=rng, detrend=False)
            periods, _power = lombscargle_periodogram(x, y_detrended, detrend=False)
            testable_max = periods.max()
        except ValueError as e:
            angle_results.append(dict(segment=label, motor=ax, n=len(x), status=f'skipped: {e}'))
            continue
        angle_results.append(dict(
            segment=label, motor=ax, n=len(x),
            x_span_deg=round(float(x.max() - x.min()), 2),
            peak_period_deg=round(float(peak_period), 3),
            peak_power=round(float(peak_power), 3),
            fap=fap,
            testable_max_deg=round(float(testable_max), 2),
            near_edge=bool(peak_period > 0.9 * testable_max),
        ))

angle_df = pd.DataFrame(angle_results)
angle_df


## Time-domain periodogram per motor, per segment (secondary cross-check, not the hypothesis)

`x = t_local_sec / 60` (minutes), `y = theta_true_i` -- here `x` and `y` are genuinely different quantities, so `lombscargle_periodogram()`'s own linear detrend (`detrend=True`) is exactly the right, non-degenerate choice, same convention `analyse_pec.ipynb`'s "Right Ascension/Declination Period" cells already use. Kept only as a cross-check: on an alt-az mount the physically meaningful domain for a worm-gear periodic error is motor angle, not wall-clock time (see the intro cell), so a mismatch between this and the angle-domain result above isn't itself evidence against the hypothesis.

In [ ]:
time_results = []
for label, seg in segments:
    t_min = seg['t_local_sec'].values / 60
    for i, ax in enumerate(AXES, start=1):
        y = seg[f'theta_true_{i}'].values
        try:
            fap, peak_period, peak_power = periodogram_false_alarm_probability(
                t_min, y, n_shuffles=FAP_N_SHUFFLES, rng=rng, detrend=True)
            periods, _power = lombscargle_periodogram(t_min, y, detrend=True)
            testable_max = periods.max()
        except ValueError as e:
            time_results.append(dict(segment=label, motor=ax, n=len(t_min), status=f'skipped: {e}'))
            continue
        time_results.append(dict(
            segment=label, motor=ax, n=len(t_min),
            t_span_min=round(float(t_min.max() - t_min.min()), 1),
            peak_period_min=round(float(peak_period), 2),
            peak_power=round(float(peak_power), 3),
            fap=fap,
            testable_max_min=round(float(testable_max), 1),
        ))

time_df = pd.DataFrame(time_results)
time_df


# Angle-domain periodogram plots (visual inspection)

In [ ]:
SIG_THRESHOLD = 0.05

fig = make_subplots(rows=len(AXES), cols=1, shared_xaxes=False,
    subplot_titles=[f'{ax}: angle-domain periodogram (all segments)' for ax in AXES],
    vertical_spacing=0.08)

for label, seg in segments:
    t = seg['t_local_sec'].values
    for i, ax in enumerate(AXES, start=1):
        x = seg[f'theta_true_{i}'].values
        y_detrended = x - np.polyval(np.polyfit(t, x, 1), t)
        try:
            periods, power = lombscargle_periodogram(x, y_detrended, detrend=False)
        except ValueError:
            continue
        fig.add_trace(go.Scatter(x=periods, y=np.log10(power + 1e-12), mode='lines',
            name=f'{label} {ax}'), row=i, col=1)

fig.update_xaxes(title_text='Period (degrees of motor rotation)', row=len(AXES), col=1)
for i in range(1, len(AXES) + 1):
    fig.update_yaxes(title_text='log10(power)', row=i, col=1)
fig.update_layout(height=900, width=1200, template='plotly_dark',
    title=f'Angle-domain periodogram per motor -- source: {source}',
    hovermode='x unified')
fig.show()


# Time-domain periodogram plots (visual inspection)

In [ ]:
fig = make_subplots(rows=len(AXES), cols=1, shared_xaxes=False,
    subplot_titles=[f'{ax}: time-domain periodogram (all segments)' for ax in AXES],
    vertical_spacing=0.08)

for label, seg in segments:
    t_min = seg['t_local_sec'].values / 60
    for i, ax in enumerate(AXES, start=1):
        y = seg[f'theta_true_{i}'].values
        try:
            periods, power = lombscargle_periodogram(t_min, y, detrend=True)
        except ValueError:
            continue
        fig.add_trace(go.Scatter(x=periods, y=np.log10(power + 1e-12), mode='lines',
            name=f'{label} {ax}'), row=i, col=1)

fig.update_xaxes(title_text='Period (minutes)', row=len(AXES), col=1)
for i in range(1, len(AXES) + 1):
    fig.update_yaxes(title_text='log10(power)', row=i, col=1)
fig.update_layout(height=900, width=1200, template='plotly_dark',
    title=f'Time-domain periodogram per motor -- source: {source}',
    hovermode='x unified')
fig.show()


# Notes

See `docs/pec_theta_space_plan.md` for the full validation history, methodology corrections (the `resid`-is-PEC-contaminated fix, the sync-guide-contamination fix, the degenerate self-paired-periodogram fix this notebook's angle-domain cell already applies), and results from every session analyzed so far. As of the last update: M1≈M3 (distinct from M2) is a consistent, repeatable pattern across every session tested, but the specific angle-domain period is *not* yet stable -- it clusters near 5-9° in several shorter/sparser segments but comes out at 34-46° in the two longest, densest segments analyzed so far. Open question: harmonic aliasing (a shorter segment latching onto a higher harmonic of the same true fundamental period a longer one resolves), vs. two genuinely different real periodicities -- not yet distinguished.